In [3]:
from decouple import config
from openai import OpenAI
from dotenv import load_dotenv
from pathlib import Path
load_dotenv(Path('.') / '.env')

True

In [7]:
OPENAI_API_KEY = config("OPENAI_API_KEY")
UPSTASH_VECTOR_REST_URL = config("UPSTASH_VECTOR_REST_URL")
UPSTASH_VECTOR_REST_TOKEN = config("UPSTASH_VECTOR_REST_TOKEN")

In [8]:
client = OpenAI(api_key=OPENAI_API_KEY)

In [9]:
def get_embedding(text, model="text-embedding-3-small"):
   text = text.replace("\n", " ")
   return client.embeddings.create(input = [text], model=model).data[0].embedding

In [10]:
documents = [
    "The cat jumped over the dog",
    "The cow jumped over the moon",
    "The turkey ran in circles",
]

In [11]:
embeddings = [get_embedding(x) for x in documents]

In [12]:
dataset = {}
for i, embedding in enumerate(embeddings):
    dataset[i] = embedding

In [13]:
from upstash_vector import Vector

from upstash_vector import Index

index = Index(url=UPSTASH_VECTOR_REST_URL, token=UPSTASH_VECTOR_REST_TOKEN)

In [14]:
vectors = []
for key, value in dataset.items():
    print(key)
    my_id = key
    embedding = value
    vectors.append(Vector(id=my_id, vector=embedding))

0
1
2


In [15]:
vectors

[Vector(id=0, vector=[-0.00186538090929389, -0.04114184156060219, -0.009518615901470184, -0.013535422272980213, 0.002213808475062251, 0.026218794286251068, -0.012074765749275684, 0.0033534253016114235, 0.005541367921978235, -0.01973104290664196, 0.01144181378185749, 0.017893049865961075, -0.036881592124700546, 0.02440514601767063, -0.02188551239669323, 0.03145281597971916, -0.02945658564567566, -0.034641917794942856, -0.08062827587127686, 0.007924064993858337, -0.024782482534646988, 0.031233718618750572, 0.006426891312003136, -0.030430356040596962, -0.005590056534856558, 0.011606138199567795, 0.028872322291135788, 0.013523250818252563, 0.014764809049665928, 0.0024161702021956444, 0.004850598983466625, -0.02677871286869049, -0.0014385951217263937, -0.051025621592998505, -0.023991292342543602, 0.009987243451178074, -0.026097074151039124, -0.023419203236699104, -0.013158085756003857, 0.0010544118704274297, -0.03023560158908367, -0.02467293292284012, 0.015227350406348705, -0.02210461162030

In [16]:
index.upsert(
  vectors=vectors
)

'Success'

In [17]:
# dataset[3] = get_embedding("The moose sat by the dog")

In [21]:
dataset[2] = get_embedding("The moose sat near the turkery")

In [22]:
# index.upsert(vectors=[Vector(id=3, vector=dataset[3])])

In [23]:
index.upsert(vectors=[Vector(id=2, vector=dataset[2])])

'Success'

In [26]:
query_str = "The moose sat near the turkery"
query_embedding = get_embedding(query_str)

In [27]:
results = index.query(
  vector=query_embedding,
  top_k=3,
  include_vectors=True,
  include_metadata=True
)

for result in results:
    print(result.id, result.score * 100)

2 100.0
3 86.58642
1 65.385574
